In [1]:
import os
from torch.utils.data import Dataset, DataLoader

DATA_FOLDER = "data"
SICK_EN_FOLDER = "sick_en"
SICK_EN_FILE = "SICK_annotated.txt"

sick_filepaths = {"en": f"./{DATA_FOLDER}/{SICK_EN_FOLDER}/{SICK_EN_FILE}",
                  "es": "unavailable"}



In [45]:
# sick_loader

class SICKDataset(Dataset):
    label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}

    def __init__(self, language, split):
        self.sentence_pairs = []
        self.labels = []
        self.original_ids = []
        filepath = sick_filepaths[language]

        self.load_sick_dataset(filepath, split)


    def load_sick_dataset(self, filepath, split):        
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")
        
        with open(filepath, "r", encoding="utf-8") as f:
            next(f) # Skip first line, since it is the column names
            for line in f:
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                    
                data = [s.strip() for s in line.split("\t")]
                
                if data[-1].lower() != split:
                    continue
                
                pair_ID = int(data[0])
                # pair_type = data[1]
                sentence_A = data[2]
                # sentence_A_expRule = data[3]
                sentence_B = data[4]
                # sentence_B_expRule = data[5]
                # relatedness_score = float(data[6])
                label = data[7].lower()
                # entailment_AB = data[8]
                # entailment_BA = data[9]
                # sentence_A_original = data[10]
                # sentence_B_original = data[11]
                # sentence_A_dataset = data[12]
                # sentence_B_datase = data[13]
                label = data[7].lower()
                
                self.sentence_pairs.append((sentence_A, sentence_B))
                self.labels.append(self.label_map[label])
                self.original_ids.append(pair_ID)
            
    def __getitem__(self, index):
        return self.sentence_pairs[index], self.labels[index]

    def __len__(self):
        return len(self.sentence_pairs)
    

In [46]:
sick_en_filepath = f"./{DATA_FOLDER}/{SICK_EN_FOLDER}/{SICK_EN_FILE}"

sick_en_dataset_train = SICKDataset("en", "train")
# sick_en_dataset_test = SICKDataset("en", "test")
# sick_en_dataset_val = SICKDataset("en", "val")

sick_en_dataset_train[2]

sick_en_dataloader_train = DataLoader(sick_en_dataset_train, batch_size=1, shuffle=True)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch as t

In [5]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch as t

# MODEL_FOLDER = "models"

# olmo_filepath = f"./{MODEL_FOLDER}/olmo_model"

# device = "cuda" if t.cuda.is_available() else "cpu"

# tokenizer = AutoTokenizer.from_pretrained(olmo_filepath, local_files_only=True)
# model = AutoModelForCausalLM.from_pretrained(olmo_filepath, local_files_only=True).to(device)

# (sentence_a, sentence_b), label = sick_en_dataset_train[1]
# prompt = f"Premise: {sentence_a} Hypothesis: {sentence_b} Label:"

#     # print(f"Prompt: {prompt}\n--------------\n")

# inputs = tokenizer(prompt, return_Tensors="pt").to(device)

# outputs = model.generate(**inputs, max_new_tokens=50)
# print(f"Prompt and answer:\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")

In [5]:
from torch import Tensor
from jaxtyping import Float
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch as t

# import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix

In [6]:


class LRProbe(t.nn.Module):
    def __init__(self, d_in: int, scaler_mean: Tensor | None = None, scaler_scale: Tensor | None = None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 3, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean)
        self.register_buffer("scaler_scale", scaler_scale)

    def _normalize(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, "n d_model"]:
        """Apply StandardScaler normalization if scaler parameters are available."""
        if self.scaler_mean is not None and self.scaler_scale is not None:
            return (x - self.scaler_mean) / self.scaler_scale
        return x

    def forward(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self.net(self._normalize(x)).squeeze(-1)

    def pred(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self(x).round()

    @property
    def direction(self) -> Float[Tensor, " d_model"]:
        return self.net[0].weight.data[0]

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        C: float = 0.1,
        device: str = "cpu",
    ) -> "LRProbe":
        """
        Train an LR probe using sklearn's LogisticRegression with StandardScaler normalization.

        Args:
            acts: Activation matrix [n_samples, d_model].
            labels: Binary labels (1=true, 0=false).
            C: Inverse regularization strength (lower = stronger regularization).
                Default 0.1 (reg_coeff=10) matches the deception-detection paper's cfg.yaml.
                The repo class default is reg_coeff=1000 (C=0.001), which is stronger.
            device: Device to place the resulting probe on.
        """
        X = acts.cpu().float().numpy()
        y = labels.cpu().float().numpy()

        # Standardize features (zero mean, unit variance) before fitting, as in the paper
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # fit_intercept=False: the paper fits on normalized data so the intercept is redundant
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000, multi_class="multinomial")
        lr_model.fit(X_scaled, y)

        # Build probe with scaler parameters baked in
        scaler_mean = t.Tensor(scaler.mean_, dtype=t.float32)
        scaler_scale = t.Tensor(scaler.scale_, dtype=t.float32)
        probe = LRProbe(acts.shape[-1], scaler_mean=scaler_mean, scaler_scale=scaler_scale).to(device)
        probe.net[0].weight.data[0] = t.Tensor(lr_model.coef_[0], dtype=t.float32).to(device)

        return probe

In [7]:
MODEL_FOLDER = "models"

olmo_filepath = f"./{MODEL_FOLDER}/olmo_model"

device = "cuda" if t.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(olmo_filepath, local_files_only=True)
hf_model = AutoModelForCausalLM.from_pretrained(olmo_filepath, local_files_only=True).to(device)

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

In [9]:
activations = {}

def get_activation(name):
    def hook(model, input, output):
        # 'output' is a tuple for some models; we want the first element (the tensor)
        if isinstance(output, tuple):
            activations[name] = output[0].detach()
        else:
            activations[name] = output.detach()
    return hook

# Identify the layer you want (e.g., layer 12)
layer_to_probe = 12

# Register the hook to the 'post-residual' part of that layer
# Note: For OLMo 3, the path is usually hf_model.model.layers[idx]
hook_handle = hf_model.model.layers[layer_to_probe].register_forward_hook(
    get_activation(f"layer_{layer_to_probe}")
)



In [17]:
(sentence_a, sentence_b), label = sick_en_dataset_train[1]
prompt = f"Premise: {sentence_a} Hypothesis: {sentence_b} Label:"

tokens = tokenizer(prompt, return_tensors="pt").to(device)
print(tokens)

# Run the model
with t.no_grad():
    hf_model(**tokens)

# Now 'activations' contains the data!
# Shape: [batch, sequence_length, d_model]
raw_acts = activations[f"layer_{layer_to_probe}"]

# Get the activation for the VERY LAST token (the one predicting the label)
# Shape: [d_model]
last_token_act = raw_acts[0, -1, :]

# Clean up the hook when done to save memory
hook_handle.remove()

{'input_ids': tensor([[42562,  1082,    25,   362,  1912,   315,  6980,   374,  5737,   304,
           264, 20085,   323,   459,  2362,   893,   374, 11509,   304,   279,
          4092, 39515,    78, 13491,    25,   362,  1912,   315, 13305,   304,
           264, 20085,   374,  5737,   323,   264,   893,   374, 11509,   304,
           279,  4092,  9587,    25]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [ ]:
from tqdm import tqdm

ACTIVATIONS_PATH = f"./data/activations/"

def get_activation(name):
    def hook(model, input, output):
        # 'output' is a tuple for some models; we want the first element (the tensor)
        if isinstance(output, tuple):
            activations[name] = output[0].detach()
        else:
            activations[name] = output.detach()
    return hook

def generate_activations(layer_to_probe, model_name, save_to_disk=True, amount_to_generate=None):
    all_acts = []
    all_labels = []

    # 1. Register Hook
    hook_handle = hf_model.model.layers[layer_to_probe].register_forward_hook(
        get_activation(f"layer_{layer_to_probe}")
    )
    
    # 2. Extraction Loop
    try:
        # Using tqdm for a progress bar
        for (sentence_a, sentence_b), label in tqdm(sick_en_dataloader_train, desc=f"Probing Layer {layer_to_probe}"):
            prompt = f"Premise: {sentence_a} Hypothesis: {sentence_b} Label:"
            tokens = tokenizer(prompt, return_tensors="pt").to(device)
        
            with t.no_grad():
                hf_model(**tokens)

            # Extract last token and move to CPU to save GPU memory
            # raw_acts shape: [1, seq_len, d_model]
            act = activations[f"layer_{layer_to_probe}"][0, -1, :].cpu()
            
            all_acts.append(act)
            all_labels.append(label)

            if amount_to_generate and len(all_acts) >= amount_to_generate:
                break
    finally:
        hook_handle.remove()

    # 3. Format into Tensors
    # X shape: [n_samples, d_model]
    X = t.stack(all_acts)
    # y shape: [n_samples]
    y = t.tensor(all_labels)

    if save_to_disk:
        # 4. Save to disk
        data_to_save = {
            "activations": X,
            "labels": y,
            "metadata": {
                "layer": layer_to_probe,
                "model": model_name
            }
        }

        save_path = f"{ACTIVATIONS_PATH}/{model_name}/{layer_to_probe}"

        t.save(data_to_save, save_path)
        print(f"Saved {len(all_acts)} samples to {save_path}")

    return X, y

In [51]:
generate_activations(1, "olmo_model", amount_to_generate=10)

Probing Layer 1:   0%|          | 9/4439 [00:07<59:04,  1.25it/s]  

Saved 10 samples to ./data/activations//olmo_model/1


(tensor([[-0.0233,  0.0087,  0.0210,  ...,  0.0095, -0.0178,  0.0051],
         [-0.0223,  0.0095,  0.0139,  ..., -0.0005, -0.0001,  0.0099],
         [-0.0195,  0.0129,  0.0159,  ..., -0.0003, -0.0025,  0.0073],
         ...,
         [-0.0200,  0.0146,  0.0154,  ..., -0.0020, -0.0021,  0.0043],
         [-0.0219,  0.0096,  0.0166,  ..., -0.0009, -0.0013,  0.0087],
         [-0.0219,  0.0135,  0.0156,  ...,  0.0030, -0.0048,  0.0067]],
        dtype=torch.bfloat16),
 tensor([0, 1, 2, 0, 1, 1, 1, 2, 1, 1]))

In [ ]:
generate_activations(2, amount_to_generate=10)

[tensor([-0.0427,  0.0269,  0.0212,  ..., -0.0083, -0.0136, -0.0036],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0437,  0.0232,  0.0254,  ..., -0.0118, -0.0129, -0.0012],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0425,  0.0425,  0.0322,  ...,  0.0001, -0.0093, -0.0061],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0442,  0.0300,  0.0374,  ..., -0.0041, -0.0100,  0.0027],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0454,  0.0267,  0.0349,  ..., -0.0051, -0.0105,  0.0007],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0439,  0.0569,  0.0430,  ...,  0.0072, -0.0099, -0.0204],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0439,  0.0403,  0.0309,  ..., -0.0069, -0.0021, -0.0171],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0437,  0.0413,  0.0302,  ..., -0.0037, -0.0065, -0.0094],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([-0.0459,  0.0432,  0.0269,  ...,  0.0021, -0.00

In [44]:
sick_en_dataset_train[0]

(('A group of kids is playing in a yard and an old man is standing in the background',
  'A group of boys in a yard is playing and a man is standing in the background'),
 'neutral')